#### Import packages

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from ipywidgets import interact, IntSlider
from data_manip.extraction.telemac_file import TelemacFile
from data_manip.formats.regular_grid import interpolate_on_grid

#### Read selafin file

In [2]:
#res = TelemacFile("r2d_ritter-hllc.slf")
res = TelemacFile("r2d_gouttedo_v1p0.slf")
x, y, tri = res.meshx, res.meshy, res.tri
times = np.array(res.times)

#### grid interpolation

In [3]:
gx = np.linspace(x.min(), x.max(), 128)
gy = np.linspace(y.min(), y.max(), 64)
grid = np.meshgrid(gx, gy)

#### visualization

In [4]:
def show_t(it=0):
    H, _ = interpolate_on_grid(tri, res.get_data_value("WATER DEPTH", it), grid=grid)
    U, _ = interpolate_on_grid(tri, res.get_data_value("VELOCITY U", it), grid=grid)
    V, _ = interpolate_on_grid(tri, res.get_data_value("VELOCITY V", it), grid=grid)
    
    H, U, V = map(lambda arr: np.nan_to_num(np.ma.filled(arr, np.nan)), [H, U, V])
    
    plt.figure(figsize=(10, 4))
    plt.pcolormesh(grid[0], grid[1], H, shading="auto", cmap="viridis")
    plt.colorbar(label="Water depth H")
    plt.quiver(grid[0][::4, ::4], grid[1][::4, ::4], U[::4, ::4], V[::4, ::4], color="white")
    plt.title(f"t={times[it]:.3f}")
    plt.xlabel("x"); plt.ylabel("y")
    plt.show()

interact(show_t, it=IntSlider(min=0, max=len(times)-1, step=1, value=0, continuous_update=False))

interactive(children=(IntSlider(value=0, continuous_update=False, description='it', max=20), Output()), _dom_c…

<function __main__.show_t(it=0)>

#### data preparation for CNN

In [5]:
def get_H_on_grid(it):
    Ht = res.get_data_value("WATER DEPTH", it)
    Ht, _ = interpolate_on_grid(tri, Ht, grid=grid)
    return np.nan_to_num(np.ma.filled(Ht, np.nan))

MAX_T = len(times)
H_all = np.stack([get_H_on_grid(it) for it in range(MAX_T)], axis=0)
H_mean, H_std = H_all.mean(), H_all.std() + 1e-8
Hn = (H_all - H_mean) / H_std

N_IN = 4
X = np.stack([Hn[i:i+N_IN] for i in range(len(Hn)-N_IN)], axis=0)
Y = np.stack([Hn[i+N_IN] for i in range(len(Hn)-N_IN)], axis=0)

split = int(0.8*len(X))
Xtr = torch.tensor(X[:split], dtype=torch.float32)
Ytr = torch.tensor(Y[:split, None], dtype=torch.float32)
Xva = torch.tensor(X[split:], dtype=torch.float32)
Yva = torch.tensor(Y[split:, None], dtype=torch.float32)

#### simple CNN with 3 layers

In [6]:
DEVICE = "cpu"
Xtr, Ytr = Xtr.to(DEVICE), Ytr.to(DEVICE)
Xva, Yva = Xva.to(DEVICE), Yva.to(DEVICE)

model = nn.Sequential(
    nn.Conv2d(N_IN, 16, 3, padding=1, padding_mode="reflect"), nn.ReLU(),
    nn.Conv2d(16, 16, 3, padding=1, padding_mode="reflect"), nn.ReLU(),
    nn.Conv2d(16, 1, 3, padding=1, padding_mode="reflect")
).to(DEVICE)

opt = torch.optim.Adam(model.parameters(), lr=1e-3)

#### traning loop

In [7]:
for epoch in range(15):
    model.train()
    pred_train = model(Xtr)
    loss_train = F.mse_loss(pred_train, Ytr)
    opt.zero_grad(); loss_train.backward(); opt.step()
    
    model.eval()
    with torch.no_grad():
        pred_val = model(Xva)
        loss_val = F.mse_loss(pred_val, Yva)
    if epoch % 2 == 0:
        print(f"Epoch {epoch:03d} | Train RMSE {torch.sqrt(loss_train):.4f} | Val RMSE {torch.sqrt(loss_val):.4f}")

Epoch 000 | Train RMSE 0.9566 | Val RMSE 0.9406
Epoch 002 | Train RMSE 0.9301 | Val RMSE 0.9197
Epoch 004 | Train RMSE 0.9004 | Val RMSE 0.8943
Epoch 006 | Train RMSE 0.8660 | Val RMSE 0.8635
Epoch 008 | Train RMSE 0.8262 | Val RMSE 0.8273
Epoch 010 | Train RMSE 0.7818 | Val RMSE 0.7862
Epoch 012 | Train RMSE 0.7363 | Val RMSE 0.7410
Epoch 014 | Train RMSE 0.6964 | Val RMSE 0.6950


#### CNN visualisation comparison with the true simulation

In [8]:
import ipywidgets as widgets

@torch.no_grad()
def show_pred_frame(it=0):
    pred = model(Xva[it:it+1]).cpu().numpy()[0,0]
    true = Yva[it:it+1].cpu().numpy()[0,0]

    # scalling
    pred = pred * H_std + H_mean
    true = true * H_std + H_mean
    err = abs(true - pred)
    
    vmin = min(true.min(), pred.min())
    vmax = max(true.max(), pred.max())

    plt.figure(figsize=(12,3))
    plt.subplot(1,3,1)
    plt.imshow(true, origin="lower", cmap="viridis", vmin=vmin, vmax=vmax)
    plt.title(f"True H(t+1) | frame {it}")
    plt.colorbar()

    plt.subplot(1,3,2)
    plt.imshow(pred, origin="lower", cmap="viridis", vmin=vmin, vmax=vmax)
    plt.title("CNN")
    plt.colorbar()

    plt.subplot(1,3,3)
    plt.imshow(err, origin="lower", cmap="Reds", vmin=0)
    plt.title("Error")
    plt.colorbar()

    plt.tight_layout()
    plt.show()

interact(
    show_pred_frame,
    it=widgets.IntSlider(min=0, max=len(Xva)-1, step=1, value=0, continuous_update=False)
)

interactive(children=(IntSlider(value=0, continuous_update=False, description='it', max=3), Output()), _dom_cl…

<function __main__.show_pred_frame(it=0)>